# 03 Inferensi ulang: kode diacak dan kode tetangga (tanpa training ulang)
Checkpoint FULL yang sudah ada (Seq2seq, CodeBERT-Transformer, CodeBERT-BART, CodeT5-small) di-decode ulang dengan kode test diganti:
kode milik contoh lain (shuffled) atau kode training paling mirip menurut BM25 (neighbour). Varian FULL re-decode memeriksa pipeline.

Argumen Seq2seq dan CodeT5 sama persis dengan perintah aslinya. Argumen CodeBERT dan BART direkonstruksi dari results.json run yang ada;
--group dan --protocol hanya label W&B. Varian FULL re-decode di sini tanpa --historical (pemeriksaan kecocokan dengan prediksi lama tidak dijalankan).

## Setup
Pakai kernel Python dari ~/venvs/dq (venv yang menjalankan semua skrip). Sel ini menyamakan lingkungan dengan bg.sh:
cwd di codeqa_exp/code, pylib_metrics di path, Java untuk METEOR, dan GPU 0.

In [ ]:
import os, sys, json
from pathlib import Path

HOME = Path.home()
EXP = HOME / "inahuy/kacuy/codeqa_exp"
CODE = EXP / "code"
os.chdir(CODE)  # scripts import kc_common / registry from here
for p in (str(CODE), str(EXP / "pylib_metrics")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{EXP / 'pylib_metrics'}:{os.environ.get('PYTHONPATH', '')}"
os.environ["PATH"] = f"{HOME / 'envs/kc_tools/bin'}:{os.environ['PATH']}"  # Java for METEOR
os.environ["JAVA_HOME"] = str(HOME / "envs/kc_tools")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("cwd:", os.getcwd(), "| python:", sys.executable)

## infer.py

In [ ]:
# Default 'smoke' (32 contoh).
VARIANTS = {
    'smoke': ['--model', 'codet5', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_full_bertsel', '--condition', 'shuffled', '--decoding', 'beam10', '--run-tag', 'smoke_codet5_full_bertsel__shuffled', '--group', 'E2', '--protocol', 'P', '--pair-tag', 'codet5_full_bertsel', '--ckpt', 'best_bertscore', '--batch', '16', '--limit', '32'],
    'seq2seq_full_redecode': ['--model', 'seq2seq', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/seq2seq_paper_results/paper_s2s_full_v2_7e0a3fc43cbf', '--condition', 'full', '--decoding', 'beam4', '--run-tag', 'seq2seq_full_v2__full_redecode', '--group', 'FULL_redecode', '--protocol', 'S', '--pair-tag', 'paper_s2s_full_v2_7e0a3fc43cbf', '--batch', '32'],
    'seq2seq_shuffled': ['--model', 'seq2seq', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/seq2seq_paper_results/paper_s2s_full_v2_7e0a3fc43cbf', '--condition', 'shuffled', '--decoding', 'beam4', '--run-tag', 'seq2seq_full_v2__shuffled', '--group', 'E2', '--protocol', 'S', '--pair-tag', 'seq2seq_full_v2__full_redecode', '--batch', '32'],
    'seq2seq_neighbour': ['--model', 'seq2seq', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/seq2seq_paper_results/paper_s2s_full_v2_7e0a3fc43cbf', '--condition', 'neighbour', '--decoding', 'beam4', '--run-tag', 'seq2seq_full_v2__neighbour', '--group', 'E3', '--protocol', 'S', '--pair-tag', 'seq2seq_full_v2__full_redecode', '--batch', '32'],
    'codebert_full_redecode': ['--model', 'codebert', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/codebert_paper_results/paper_v1_67d4ca132a23', '--condition', 'full', '--decoding', 'beam10', '--run-tag', 'codebert_t__full_redecode', '--group', 'FULL_redecode', '--protocol', 'P', '--pair-tag', 'paper_v1_67d4ca132a23'],
    'codebert_shuffled': ['--model', 'codebert', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/codebert_paper_results/paper_v1_67d4ca132a23', '--condition', 'shuffled', '--decoding', 'beam10', '--run-tag', 'codebert_t__shuffled', '--group', 'E2', '--protocol', 'P', '--pair-tag', 'codebert_t__full_redecode'],
    'codebert_neighbour': ['--model', 'codebert', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/codebert_paper_results/paper_v1_67d4ca132a23', '--condition', 'neighbour', '--decoding', 'beam10', '--run-tag', 'codebert_t__neighbour', '--group', 'E3', '--protocol', 'P', '--pair-tag', 'codebert_t__full_redecode'],
    'bart_full_greedy': ['--model', 'bart', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/codebert_bart_results/vps_v1_b1bb1511e498', '--condition', 'full', '--decoding', 'greedy', '--run-tag', 'bart__full_greedy_redecode', '--group', 'FULL_redecode', '--protocol', 'P-deviant', '--pair-tag', 'vps_v1_b1bb1511e498'],
    'bart_shuffled_greedy': ['--model', 'bart', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/codebert_bart_results/vps_v1_b1bb1511e498', '--condition', 'shuffled', '--decoding', 'greedy', '--run-tag', 'bart__shuffled_greedy', '--group', 'E2', '--protocol', 'P-deviant', '--pair-tag', 'bart__full_greedy_redecode'],
    'bart_neighbour_greedy': ['--model', 'bart', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/codebert_bart_results/vps_v1_b1bb1511e498', '--condition', 'neighbour', '--decoding', 'greedy', '--run-tag', 'bart__neighbour_greedy', '--group', 'E3', '--protocol', 'P-deviant', '--pair-tag', 'bart__full_greedy_redecode'],
    'bart_full_beam10': ['--model', 'bart', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/codebert_bart_results/vps_v1_b1bb1511e498', '--condition', 'full', '--decoding', 'beam10', '--run-tag', 'bart__full_beam10', '--group', 'E4', '--protocol', 'P', '--pair-tag', 'vps_v1_b1bb1511e498'],
    'bart_shuffled_beam10': ['--model', 'bart', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/codebert_bart_results/vps_v1_b1bb1511e498', '--condition', 'shuffled', '--decoding', 'beam10', '--run-tag', 'bart__shuffled_beam10', '--group', 'E4', '--protocol', 'P', '--pair-tag', 'bart__full_beam10'],
    'bart_neighbour_beam10': ['--model', 'bart', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/prior_runs/kc-codeqa-reproduce/codebert_bart_results/vps_v1_b1bb1511e498', '--condition', 'neighbour', '--decoding', 'beam10', '--run-tag', 'bart__neighbour_beam10', '--group', 'E4', '--protocol', 'P', '--pair-tag', 'bart__full_beam10'],
    'codet5_shuffled': ['--model', 'codet5', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_full_v2', '--condition', 'shuffled', '--decoding', 'beam10', '--run-tag', 'codet5_full_v2__shuffled', '--group', 'E2', '--protocol', 'P', '--pair-tag', 'codet5_full_v2', '--batch', '16'],
    'codet5_neighbour': ['--model', 'codet5', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_full_v2', '--condition', 'neighbour', '--decoding', 'beam10', '--run-tag', 'codet5_full_v2__neighbour', '--group', 'E3', '--protocol', 'P', '--pair-tag', 'codet5_full_v2', '--batch', '16'],
    'codet5_bertsel_shuffled': ['--model', 'codet5', '--run-dir', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/codet5_full_bertsel', '--condition', 'shuffled', '--decoding', 'beam10', '--run-tag', 'codet5_full_bertsel__shuffled', '--group', 'E2', '--protocol', 'P', '--pair-tag', 'codet5_full_bertsel', '--ckpt', 'best_bertscore', '--batch', '16'],
}
RUN = 'smoke'  # ganti ke salah satu kunci VARIANTS
ARGS = VARIANTS[RUN]
sys.argv = ['infer.py', *ARGS]
print(RUN, ARGS)

**Sumber:** code/infer.py

```text
Inference-only conditions (E2 shuffled, E3 neighbour, E4 re-decodes, FULL re-decode checks).

Each model keeps its FULL template, truncation, precision and decoding; only the code string changes.
A FULL re-decode is compared with the historical test predictions to validate the pipeline.
```

In [ ]:
import argparse
import json
import os
import sys
import time
from pathlib import Path

import torch

import kc_common as kc

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("--model", choices=("seq2seq", "codebert", "bart", "codet5"), required=True)
parser.add_argument("--run-dir", required=True, help="FULL run directory holding the checkpoint")
parser.add_argument("--condition", choices=("full", "shuffled", "neighbour"), required=True)
parser.add_argument("--decoding", choices=("beam4", "beam10", "greedy", "greedy_hf"), required=True)
parser.add_argument("--split", choices=("test", "dev"), default="test")
parser.add_argument("--run-tag", required=True)
parser.add_argument("--group", required=True)
parser.add_argument("--protocol", required=True)
parser.add_argument("--pair-tag", required=True, help="FULL run this condition is compared against")
parser.add_argument("--historical", default=None, help="historical predictions jsonl for FULL agreement check")
parser.add_argument("--batch", type=int, default=64)
parser.add_argument("--ckpt", default="best_bleu", help="checkpoint folder inside the FULL run")
parser.add_argument("--limit", type=int, default=0, help="smoke: first N examples only")
args = parser.parse_args(ARGS)

device = torch.device("cuda")
torch.backends.cudnn.benchmark = False  # as in the historical Seq2seq runner
torch.backends.cudnn.deterministic = True
run_dir = Path(args.run_dir)
out = kc.EXP / "runs" / (f"smoke_{args.run_tag}" if args.limit else args.run_tag)
out.mkdir(parents=True, exist_ok=True)
if args.limit:
    os.environ.setdefault("WANDB_MODE", "offline")

rows = kc.load_split(args.split)

In [ ]:
if args.condition == "full":
    codes, code_ids = [r["code"] for r in rows], [f"{args.split}:{r['id']}" for r in rows]
elif args.condition == "shuffled":
    assert args.split == "test"
    donor = json.loads((kc.EXP / "maps/shuffled_mapping.json").read_text())["donor_test_id"]
    codes, code_ids = [rows[donor[i]]["code"] for i in range(len(rows))], [f"test:{donor[i]}" for i in range(len(rows))]
else:
    assert args.split == "test"
    train = kc.load_split("train")
    nb = json.loads((kc.EXP / "maps/neighbour_mapping.json").read_text())["mapping"]
    codes = [train[nb[i]["neighbour_train_id"]]["code"] for i in range(len(rows))]
    code_ids = [f"train:{nb[i]['neighbour_train_id']}" for i in range(len(rows))]
if args.limit:
    rows, codes, code_ids = rows[:args.limit], codes[:args.limit], code_ids[:args.limit]
inputs = [{**r, "code": c} for r, c in zip(rows, codes)]  # gold answer is never used for generation

In [ ]:
def batched(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

In [ ]:
# ---------------- model families ----------------
def decode_codebert():
    import codebert_training as cbt
    from transformers import RobertaTokenizer
    assert args.decoding == "beam10"
    tokenizer = RobertaTokenizer.from_pretrained(cbt.MODEL_ID, revision=cbt.MODEL_REVISION)
    model = cbt.build_model(pretrained=False)  # weights come from best.pt
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location="cpu", weights_only=True)["model"])
    model.to(device)
    tensors = cbt.tensor_data(inputs, tokenizer, training=False)  # upstream features, source 400
    return cbt.predict(model, tensors, tokenizer, args.batch, device, log=lambda m: None)

In [ ]:
def decode_bart():
    import codebert_bart_training as cbb
    from transformers import BartTokenizer, RobertaTokenizer, GenerationConfig, BartForConditionalGeneration
    from transformers.modeling_outputs import BaseModelOutput
    settings = cbb.Settings()  # source 400, target 32, as in the FULL run
    src_tok = RobertaTokenizer.from_pretrained(cbb.MODEL_ID, revision=cbb.MODEL_REVISION)
    tgt_tok = BartTokenizer.from_pretrained(cbb.BART_ID, revision=cbb.BART_REVISION)
    model = cbb.build_model(device)
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location="cpu", weights_only=True)["model"])
    model.eval()
    ids, mask = cbb.features(inputs, src_tok, tgt_tok, settings, training=False)
    preds = []
    if args.decoding == "greedy":
        # Same greedy loop as the FULL run (cbb.Hybrid.generate under fp16 autocast), batched.
        for s in range(0, len(ids), args.batch):
            with torch.autocast("cuda", dtype=torch.float16):
                out_ids = model.generate(ids[s:s + args.batch].to(device), mask[s:s + args.batch].to(device), settings.target_length)
            preds += [tgt_tok.decode(x, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip() for x in out_ids]
        return preds
    # beam 10: the trained decoder, LM head and logits bias inside a BART wrapper so HF beam search drives it.
    bart = BartForConditionalGeneration.from_pretrained(cbb.BART_ID, revision=cbb.BART_REVISION, attn_implementation="eager")
    bart.model.decoder.load_state_dict(model.decoder.state_dict())
    bart.lm_head.weight = bart.model.decoder.embed_tokens.weight
    bart.final_logits_bias.copy_(model.logits_bias.view_as(bart.final_logits_bias))
    bart.to(device).eval()
    # greedy_hf (num_beams=1) only validates this wrapper against the historical greedy predictions.
    gen = GenerationConfig(num_beams=1 if args.decoding == "greedy_hf" else 10, max_new_tokens=settings.target_length, decoder_start_token_id=model.start_id,
                           eos_token_id=model.eos_id, pad_token_id=model.pad_id, bos_token_id=0, length_penalty=1.0,
                           early_stopping=False, no_repeat_ngram_size=0, do_sample=False)
    # Replace bart-base's own generation defaults (no_repeat_ngram_size=3, forced BOS/EOS, ...) so none leak in.
    bart.generation_config = gen
    for s in range(0, len(ids), args.batch):
        i, m = ids[s:s + args.batch].to(device), mask[s:s + args.batch].to(device)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            memory = model.encode(i, m)
            out_ids = bart.generate(encoder_outputs=BaseModelOutput(last_hidden_state=memory), attention_mask=m,
                                    generation_config=gen)
        preds += [tgt_tok.decode(x, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip() for x in out_ids]
    return preds

In [ ]:
def decode_seq2seq():
    import argparse as ap
    ref = kc.REPO / "references/NeuralCodeSum"
    sys.path.insert(0, str(ref))
    import importlib.util
    from c2nl.inputters.vocabulary import Vocabulary
    from c2nl.objects.code import Code
    from c2nl.inputters.vector import vectorize, batchify
    from c2nl.utils.copy_utils import make_src_map, collapse_copy_scores
    from c2nl.translator import Translator, TranslationBuilder, GNMTGlobalScorer
    spec = importlib.util.spec_from_file_location("codeqa_reference_model", ref / "main/model.py")
    wrapper = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(wrapper)
    assert args.decoding == "beam4"
    config = json.loads((run_dir / "config.json").read_text())
    saved_vocab = json.loads((run_dir / "vocab.json").read_text())

    def rebuild(table):
        v = Vocabulary()
        for tok, idx in sorted(table.items(), key=lambda kv: kv[1]):
            if tok not in v.tok2ind:
                v.add(tok)
            assert v.tok2ind[tok] == idx, f"vocab index mismatch for {tok!r}"
        assert len(v) == len(table)
        return v
    src_dict, tgt_dict = rebuild(saved_vocab["source"]), rebuild(saved_vocab["target"])
    margs = ap.Namespace(**config["model"])
    model = wrapper.Code2NaturalLanguage(margs, src_dict, tgt_dict)
    model.network.load_state_dict(torch.load(run_dir / "best.pt", map_location="cpu", weights_only=True)["state_dict"])
    model.cuda()
    model.network.eval()
    question_only = config.get("input_mode") == "question_only"
    assert not question_only, "ablations run on FULL checkpoints only"

    def item(row):
        code = Code(row["id"])
        code.language = 1
        tokens = ["[CLS]"] + row["question"].split() + ["[SEP]"] + row["code"].split()
        code.tokens = [t.lower() for t in tokens][:margs.max_src_len]
        code.text = " ".join(code.tokens)
        return vectorize({"code": code, "summary": None}, model)

    translator = Translator(model, True, beam_size=4, n_best=1, max_length=margs.max_tgt_len,
                            global_scorer=GNMTGlobalScorer(0, 0, "none", "none"), copy_attn=True,
                            block_ngram_repeat=3, replace_unk=True)
    builder = TranslationBuilder(model.tgt_dict, n_best=1, replace_unk=True)
    preds = []
    with torch.no_grad():
        for chunk in batched(inputs, args.batch):  # the FULL runner used 8; agreement is checked on the FULL re-decode
            batch = batchify([item(r) for r in chunk])
            beam_in = {k: batch[k] for k in ("code_word_rep", "code_char_rep", "code_type_rep", "code_mask_rep", "code_len")}
            beam_in = {k: v.to(device) if torch.is_tensor(v) else v for k, v in beam_in.items()}
            beam_in["src_map"] = make_src_map(batch["src_map"]).to(device)
            beam_in["alignment"] = None
            beam_in["blank"], beam_in["fill"] = collapse_copy_scores(model.tgt_dict, batch["src_vocab"])
            result = translator.translate_batch(beam_in)
            translations = builder.from_batch(result, batch["code_tokens"], [[]] * batch["batch_size"], batch["src_vocab"])
            preds += [" ".join(t.pred_sents[0]) for t in translations]
    return preds

In [ ]:
def decode_codet5():
    from transformers import AutoTokenizer, T5ForConditionalGeneration
    assert args.decoding == "beam10"
    cfg = json.loads((run_dir / "config.json").read_text())
    assert cfg["input_mode"] == "full", "ablations run on FULL checkpoints only"
    tok = AutoTokenizer.from_pretrained(run_dir / args.ckpt)
    model = T5ForConditionalGeneration.from_pretrained(run_dir / args.ckpt).to(device).eval()
    src = tok([kc.codet5_source(r["question"], r["code"], "full") for r in inputs], max_length=cfg["max_source"], truncation=True)["input_ids"]
    order = sorted(range(len(src)), key=lambda i: -len(src[i]))
    preds = [None] * len(src)
    for idx in batched(order, args.batch):
        width = max(len(src[i]) for i in idx)
        ids = torch.full((len(idx), width), tok.pad_token_id, dtype=torch.long)
        mask = torch.zeros_like(ids)
        for j, i in enumerate(idx):
            ids[j, :len(src[i])] = torch.tensor(src[i]); mask[j, :len(src[i])] = 1
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            gen = model.generate(input_ids=ids.to(device), attention_mask=mask.to(device), num_beams=10,
                                 max_new_tokens=cfg["generation_max_new_tokens"], do_sample=False)
        for i, text in zip(idx, tok.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=False)):
            preds[i] = text.strip()
    return preds

In [ ]:
t0 = time.time()
predictions = {"codebert": decode_codebert, "bart": decode_bart, "seq2seq": decode_seq2seq, "codet5": decode_codet5}[args.model]()
seconds = time.time() - t0
assert len(predictions) == len(rows) and all(isinstance(p, str) for p in predictions)
metrics, per = kc.evaluate(predictions, [r["answer"] for r in rows])
condition = args.condition.upper()
results = {"run_tag": out.name, "model": args.model, "condition": condition, "split": args.split,
           "decoding": args.decoding, "source_run": str(run_dir), "pair_run_tag": args.pair_tag,
           "inference_seconds": seconds, "n": len(rows), args.split: metrics}
if args.historical:
    hist = {int(r["id"]): r["prediction"] for r in kc.read_jsonl(args.historical)}
    same = sum(hist.get(r["id"]) == p for r, p in zip(rows, predictions))
    results["agreement_with_historical_predictions"] = {"file": args.historical, "identical": same, "n": len(rows),
                                                         "pct": 100.0 * same / len(rows)}
config = {"experiment": args.group, "model": args.model, "condition": condition, "protocol": args.protocol,
          "split": args.split, "decoding": args.decoding, "source_run": str(run_dir), "pair_run_tag": args.pair_tag,
          "checkpoint_sha256": kc.sha256_file(run_dir / "best.pt") if (run_dir / "best.pt").exists() else None,
          "seed": 42, "limit": args.limit, "batch": args.batch, "data_sha256": kc.data_hashes(),
          "metric_provenance": kc.codeqa_metrics.metric_provenance(kc.REPO), "torch": torch.__version__}
if args.condition == "shuffled":
    config["mapping"] = "maps/shuffled_mapping.json"
elif args.condition == "neighbour":
    config["mapping"] = "maps/neighbour_mapping.json"
kc.write_json(out / "config.json", config)
kc.write_json(out / "results.json", results)
pred_name = "test_predictions.jsonl" if args.split == "test" else "dev_predictions.jsonl"
kc.write_predictions(out / pred_name, rows, predictions, per, condition, code_ids)
run = kc.wandb_init(out, out.name, args.group, [args.model, condition, args.protocol, "seed42"], config, "inference")
if args.split == "test":
    records = kc.read_jsonl(out / pred_name)
    kc.wandb_finalize(run, out, out.name, metrics, records)
else:
    run.summary.update({f"dev/{k}": v for k, v in metrics.items() if isinstance(v, (int, float)) and not isinstance(v, bool)})
if "agreement_with_historical_predictions" in results:
    run.summary["agreement_with_historical_pct"] = results["agreement_with_historical_predictions"]["pct"]

In [ ]:
run.finish()
print("DONE", out.name, json.dumps({k: round(metrics[k], 3) for k in ("BLEU", "F1", "EM", "ROUGE_L", "METEOR")}),
      json.dumps(results.get("agreement_with_historical_predictions", {})), f"{seconds:.0f}s", flush=True)

In [ ]:
# Opsional untuk run panjang: jalankan varian terpilih di tmux lewat bg.sh (log di logs/<RUN>.log),
# supaya training tidak mati kalau browser/kernel putus. Hapus tanda # untuk memakai.
# import subprocess
# subprocess.run([str(EXP / "bg.sh"), RUN, sys.executable, 'infer.py', *ARGS], check=True)

### Ringkasan hasil test (dibaca dari runs/<tag>/results.json)

In [ ]:
TAGS = ['smoke_codet5_full_bertsel__shuffled', 'seq2seq_full_v2__full_redecode', 'seq2seq_full_v2__shuffled', 'seq2seq_full_v2__neighbour', 'codebert_t__full_redecode', 'codebert_t__shuffled', 'codebert_t__neighbour', 'bart__full_greedy_redecode', 'bart__shuffled_greedy', 'bart__neighbour_greedy', 'bart__full_beam10', 'bart__shuffled_beam10', 'bart__neighbour_beam10', 'codet5_full_v2__shuffled', 'codet5_full_v2__neighbour', 'codet5_full_bertsel__shuffled']
for tag in TAGS:
    for cand in (tag, "smoke_" + tag):
        f = EXP / "runs" / cand / "results.json"
        if f.exists():
            r = json.loads(f.read_text())
            t = r.get("test", {})
            print(cand, {k: round(v, 2) for k, v in t.items() if isinstance(v, (int, float))})
            break
    else:
        print(tag, "-> belum ada results.json")